In [1]:
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
import torch.optim as optim
import torch.nn as nn
import torch

import sentencepiece as spm

import math

import tqdm

## Hyper-parameters

In [2]:
PAD_ID, SOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

# model & dataset paths
SRC_FILE = "corpus/train.skz"
TGT_FILE = "corpus/train.zh"
SP_SRC_MODEL = "models/sp_merged_skz.model"
SP_TGT_MODEL = "models/sp_merged_zh.model"
SAVE_PATH = "models/model.pt"
CHECKPOINT = "models/checkpoint.pt"

# dataloader
# DATA_LOADER_BATCH_SIZE = 32
MAX_SEQ_LEN = 128
NUM_WORKERS = 4

# model
D_MODEL = 256
NHEAD = 4
NUM_LAYERS = 4
DIM_FEEDFORWARD = 1024
DROPOUT = 0.1
SP_MAX_LEN = 5000

# train
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-3
WARMUP_STEPS = 4000
BETAS = (0.9, 0.98)
EPS = 1e-9

## Load data

In [3]:
src_sp = spm.SentencePieceProcessor()
src_sp.Load(model_file=SP_SRC_MODEL)
tgt_sp = spm.SentencePieceProcessor()
tgt_sp.Load(model_file=SP_TGT_MODEL)

class TranslationDataset(Dataset):
    def __init__(self, max_len=MAX_SEQ_LEN):
        self.src_lines = open(SRC_FILE, encoding="utf-8").readlines()
        self.tgt_lines = open(TGT_FILE, encoding="utf-8").readlines()
        self.max_len = max_len

    def __len__(self): return len(self.src_lines)

    def __getitem__(self, idx):
        # 密文走 src_sp，中文走 tgt_sp
        s = [SOS_ID] + src_sp.encode(self.src_lines[idx].strip())[:self.max_len-2] + [EOS_ID]
        t = [SOS_ID] + tgt_sp.encode(self.tgt_lines[idx].strip())[:self.max_len-2] + [EOS_ID]
        return torch.tensor(s, dtype=torch.long), torch.tensor(t, dtype=torch.long)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_pad = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=PAD_ID)
    tgt_pad = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_ID)
    return src_pad, tgt_pad  # 仅返回2个值，匹配训练循环

## Model

In [4]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=SP_MAX_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=D_MODEL, nhead=NHEAD, num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, dim_feedforward=DIM_FEEDFORWARD, dropout=dropout, batch_first=True
        )
        self.out = nn.Linear(d_model, tgt_vocab)
        self.d_model = d_model

    def forward(self, src, tgt, src_pad_mask=None, tgt_pad_mask=None, tgt_mask=None):
        src = self.pos_enc(self.src_emb(src) * math.sqrt(self.d_model))
        tgt = self.pos_enc(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        memory = self.transformer.encoder(src, src_key_padding_mask=src_pad_mask)
        out = self.transformer.decoder(
            tgt, memory, 
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad_mask, 
            memory_key_padding_mask=src_pad_mask
        )
        return self.out(out)

## Predict

In [5]:
def translate(model, cipher_text, src_sp, tgt_sp, max_len=64):
    model.eval()
    device = next(model.parameters()).device
    src_ids = torch.tensor([SOS_ID] + src_sp.encode(cipher_text) + [EOS_ID], dtype=torch.long).unsqueeze(0).to(device)
    tgt_ids = torch.tensor([SOS_ID], dtype=torch.long).unsqueeze(0).to(device)
    src_pad = src_ids == PAD_ID

    for _ in range(max_len - 1):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_ids.size(1)).to(device)
        tgt_pad = tgt_ids == PAD_ID
        out = model(src_ids, tgt_ids, src_pad_mask=src_pad, tgt_pad_mask=tgt_pad, tgt_mask=tgt_mask)
        next_token = out[:, -1, :].argmax(dim=-1)
        tgt_ids = torch.cat([tgt_ids, next_token.unsqueeze(0)], dim=1)
        if next_token.item() == EOS_ID: break
    return tgt_sp.decode(tgt_ids.squeeze().tolist()[1:-1])

def beam_search(model, src_text, src_sp, tgt_sp, beam_size=5, max_len=64, length_penalty=0.6):
    model.eval()
    device = next(model.parameters()).device
    
    # 1. 编码源端
    src_ids = torch.tensor([SOS_ID] + src_sp.encode(src_text) + [EOS_ID], dtype=torch.long).unsqueeze(0).to(device)
    src_pad_mask = (src_ids == PAD_ID)
    
    # 2. 初始 Beam
    beams = [{"seq": [SOS_ID], "score": 0.0, "finished": False}]
    
    for step in range(1, max_len):
        active = [b for b in beams if not b["finished"]]
        if not active:
            break
            
        # 批量构建 tgt 序列
        batch_seqs = [torch.tensor(b["seq"], dtype=torch.long, device=device).unsqueeze(0) for b in active]
        seqs_tensor = torch.cat(batch_seqs, dim=0)
        tgt_pad_mask = (seqs_tensor == PAD_ID)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(seqs_tensor.size(1)).to(device).bool()
        
        # 扩展 src 匹配 Batch 大小
        src_exp = src_ids.expand(seqs_tensor.size(0), -1)
        src_pad_exp = src_pad_mask.expand(seqs_tensor.size(0), -1)
        
        with torch.no_grad():
            out = model(src_exp, seqs_tensor, tgt_mask=tgt_mask, 
                        src_pad_mask=src_pad_exp, tgt_pad_mask=tgt_pad_mask)
            
        # 获取最后一步 log_prob
        log_probs = torch.log_softmax(out[:, -1, :], dim=-1)
        
        # 收集所有候选
        candidates = []
        for i, b in enumerate(active):
            for token_id in range(log_probs.size(1)):
                prob = log_probs[i, token_id].item()
                new_score = b["score"] + prob
                
                # 长度惩罚
                if length_penalty != 0:
                    new_score /= ((5 + len(b["seq"])) / 6) ** length_penalty
                    
                new_seq = b["seq"] + [token_id]
                finished = token_id == EOS_ID
                
                candidates.append({"seq": new_seq, "score": new_score, "finished": finished})
                
        # 保留 top K
        candidates.sort(key=lambda x: x["score"], reverse=True)
        beams = candidates[:beam_size]
        
    # 返回最优序列 (去掉 SOS)
    best_seq = beams[0]["seq"][1:]
    if best_seq and best_seq[-1] == EOS_ID:
        best_seq = best_seq[:-1]
        
    return tgt_sp.decode(best_seq)

## Train
### Load data

In [6]:
dataset = TranslationDataset()
loader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn, 
    # num_workers=NUM_WORKERS, 
    # pin_memory=True
)

### Load model

In [7]:
model = Seq2SeqTransformer(
    src_vocab=src_sp.vocab_size(),
    tgt_vocab=tgt_sp.vocab_size(),
    d_model=D_MODEL, nhead=NHEAD, num_layers=NUM_LAYERS, dropout=DROPOUT
)

### Main loop

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = optim.Adam(model.parameters(), lr=LR, betas=BETAS, eps=1e-9)
scheduler = LambdaLR(optimizer, lr_lambda=lambda step: min((step+1)**-0.5, (step+1)*WARMUP_STEPS**-1.5))

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for src, tgt in tqdm.tqdm(loader):
        src, tgt = src.to(device), tgt.to(device)
        tgt_in = tgt[:, :-1]
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_in.size(1)).to(device)
        tgt_mask = tgt_mask.bool()
        src_pad = src == PAD_ID
        tgt_pad = tgt_in == PAD_ID

        optimizer.zero_grad()
        out = model(src, tgt_in, src_pad_mask=src_pad, tgt_pad_mask=tgt_pad, tgt_mask=tgt_mask)
        loss = criterion(out.transpose(1, 2), tgt[:, 1:])
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")
    torch.save({
        'epoch': epoch, 
        'model_state_dict': model.state_dict(), 
        'optimizer_state_dict': opt.state_dict(), 
        'loss': avg
    }, CHECKPOINT)

torch.save(model.state_dict(), SAVE_PATH)

  0%|          | 2/18728 [00:12<32:43:05,  6.29s/it]


KeyboardInterrupt: 

### Test

In [ ]:
translate_result = translate(model, "bsbtjyrernx", src_sp, tgt_sp)
beam_search_result = beam_search(model, "bsbtjyrernx", src_sp, tgt_sp, beam_size=5, max_len=64, length_penalty=0.6)

print(translate_result)
print(beam_search_result)

d:\programs\endfield\translator\.venv\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
d:\programs\endfield\translator\.venv\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


汉促进是位于乌克兰东部的汽车模式co 足球后卫锂琛Th雁结束时拔各级文物保护单位υ模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式模式
模式签顷为唇形科斯科特模式υ模式υ模式υ模式模式雁汽车模式υ模式υ模式υ模式υ模式υ模式υ模式υ模式υ模式模式 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫 足球后卫各级文物保护单位湖南模式模式模式模式模式模式模式模式模式模式
